# Описание задачи "Разработка нефтегазовых месторождений"

Набор данных содержит 442 о различных нефтегазовых месторождениях.
Тренировочный набор - 309 строк.
Тестовый набор - 133 строк.

Каждое месторождение обладает 19 параметрами:
1. Field name - название месторождения
2. Reservoir unit - юнит месторождения
3. Country - страна расположения
4. Region - регион расположения
5. Basin name - название бассейна пород
6. Tectonic regime - тектонический режим
7. Latitude - широта
8. Longitude - долгота
9. Operator company - название компании
10. Onshore or oﬀshore - на суше или нет
11. Hydrocarbon type (main) - тип углеводорода
12. Reservoir status (current) - статус месторождения
13. Structural setting - структурные свойства
14. Depth (top reservoir ft TVD) - глубина
15. Reservoir period - литологический период
16. Lithology (main) - литология
17. Thickness (gross average ft) - общая толщина
18. Thickness (net pay average ft) - эффективная толщина
19. Porosity (matrix average.. - пористость
20. Permeability (air average mD) – проницаемость

**Что нужно сделать**:

Принять участие в соревновании на Kaggle:
Оно доступно по [ссылке](https://www.kaggle.com/competitions/classification-of-oil-and-gas/submissions#).

Разработать и оформить решение в ноутбуке:
* Исследование и анализ датасета.
* Предобработка данных.
* Feature Engineering (если необходимо).
* Подбор признаков, их анализ и оценка важности.
* Обучение нескольких моделей, их сравнение.
* Подбор гиперпараметров.
* Выбор лучшей модели и объяснение выбора.
* Предсказание на тестовых данных.


# Считывание данных

In [1]:
import opendatasets as od

# Загрузим датасет на прямую с kaggle
# Для автомтическй загрузки надо положить файл kaggle.json в папку с ноутбуком
# Файл скачивается в настройках в разделе Legacy API Credentials
dataset_url = 'https://www.kaggle.com/competitions/classification-of-oil-and-gas/data'

od.download(dataset_url)

Skipping, found downloaded files in "./classification-of-oil-and-gas" (use force=True to force download)


In [2]:
# иморитирование всех необходимых библиотек
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# Считываем тренировочные данные
train = pd.read_csv("./classification-of-oil-and-gas/train_oil.csv")
test = pd.read_csv("./classification-of-oil-and-gas/oil_test.csv")

print(f"Train dataset shape: {train.shape}")
print(f"Test dataset shape: {test.shape}")

Train dataset shape: (309, 20)
Test dataset shape: (133, 19)


In [4]:
# Объеденим в список, на случай, если нам потребуется однотиная обработка
data = [train, test]

In [5]:
# Посмотрим как выглядят данные
train.head(6)

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
0,ZHIRNOV,MELEKESKIAN,RUSSIA,FORMER SOVIET UNION,VOLGA-URAL,COMPRESSION/EVAPORITE,51.0000,44.8042,NIZHNEVOLZHSKNET,ONSHORE,OIL,DECLINING PRODUCTION,FORELAND,1870,CARBONIFEROUS,SANDSTONE,262.0,33.0,24.0,30.0
1,LAGOA PARDA,LAGOA PARDA (URUCUTUCA),BRAZIL,LATIN AMERICA,ESPIRITO SANTO,EXTENSION,-19.6017,-39.8332,PETROBRAS,ONSHORE,OIL,NEARLY DEPLETED,PASSIVE MARGIN,4843,PALEOGENE,SANDSTONE,2133.0,72.0,23.0,350.0
2,ABQAIQ,ARAB D,SAUDI ARABIA,MIDDLE EAST,THE GULF,COMPRESSION/EVAPORITE,26.0800,49.8100,SAUDI ARAMCO,ONSHORE,OIL,REJUVENATING,FORELAND,6050,JURASSIC,LIMESTONE,250.0,184.0,21.0,410.0
3,MURCHISON,BRENT,UK /NORWAY,EUROPE,NORTH SEA NORTHERN,EXTENSION,61.3833,1.7500,CNR,OFFSHORE,OIL,NEARLY DEPLETED,RIFT,8988,JURASSIC,SANDSTONE,425.0,300.0,22.0,750.0
4,WEST PEMBINA,NISKU (PEMBINA L POOL),CANADA,NORTH AMERICA,WESTERN CANADA,COMPRESSION,53.2287,-115.8008,NUMEROUS,ONSHORE,OIL,UNKNOWN,FORELAND,9306,DEVONIAN,DOLOMITE,233.0,167.0,11.8,1407.0
5,UCHKYR,XV-1,UZBEKISTAN,FORMER SOVIET UNION,AMU DARYA,INVERSION/COMPRESSION/EXTENSION/EVAPORITE,40.1494,62.9906,SREDAZGAZPROM,ONSHORE,GAS,DECLINING PRODUCTION,INVERSION/RIFT,5443,JURASSIC,DOLOMITE,82.0,59.0,16.0,61.0


# Иследование и обработка данных

## Считывание данных

In [6]:
# процент дублей 
(len(train) - len( train.drop_duplicates() )) * 100 / len(train)

0.0

Дублирующих строк нет.

In [7]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Field name                      309 non-null    str    
 1   Reservoir unit                  309 non-null    str    
 2   Country                         282 non-null    str    
 3   Region                          271 non-null    str    
 4   Basin name                      271 non-null    str    
 5   Tectonic regime                 309 non-null    str    
 6   Latitude                        282 non-null    float64
 7   Longitude                       279 non-null    float64
 8   Operator company                309 non-null    str    
 9   Onshore/Offshore                309 non-null    str    
 10  Hydrocarbon type                309 non-null    str    
 11  Reservoir status                309 non-null    str    
 12  Structural setting              309 non-null   

In [8]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Field name                      133 non-null    str    
 1   Reservoir unit                  133 non-null    str    
 2   Country                         120 non-null    str    
 3   Region                          117 non-null    str    
 4   Basin name                      125 non-null    str    
 5   Tectonic regime                 133 non-null    str    
 6   Latitude                        120 non-null    float64
 7   Longitude                       117 non-null    float64
 8   Operator company                133 non-null    str    
 9   Hydrocarbon type                133 non-null    str    
 10  Reservoir status                133 non-null    str    
 11  Structural setting              133 non-null    str    
 12  Depth                           133 non-null   

In [9]:
# Выведем версию Pandas (в 3.0 - выводит тип str, а не object)
print(f"Версия Pandas: {pd.__version__}")

Версия Pandas: 3.0.2


## Обработка пропущенных значений

Есть пропущенные значения в тренировочных и тестовых данных. Визуализируем пропуски.

In [10]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

Country       27
Region        38
Basin name    38
Latitude      27
Longitude     30
dtype: int64

In [11]:
train.isna().sum().sum()

np.int64(160)

Поле Field name (название месторождения) у нас везде заполнено и если оно будет встречаться и там, и там, то по нему мы попробуем востановить пропущенные значения.

In [12]:
def check_column_for_fill(df, check_col):
    # Разделим данные на две таблицы, где есть пропуски и где все поля заполнены
    
    # .any(axis='columns') - перебирает все строки и помечает всю строку True, если есть хотя бы один NaN в любой колонке
    missed = df[df.isna().any(axis='columns')] 
    
    filled = df.dropna()
    
    # Получим список заполненных полей    
    filled_names = filled[check_col].unique()
    
    # Отберём те строки, в таблице с пропусками, которые есть без пропусков
    common_names = missed[missed[check_col].isin(filled_names)]
    
    # Выведем результат из общей таблицы, чтобы визауально оценить возможность заполнения пропусков
    visual_check = df[df[check_col].isin(common_names[check_col])].sort_values(check_col)

    # создадим таблицу справочник с модой и медианой по названию месторождения
    mapping = filled.groupby(check_col).agg({
        'Country': lambda x: x.mode().iat[0], # так как мода вычисляется, то для указания индекса используем iat 
        'Region': lambda x: x.mode().iat[0],
        'Basin name': lambda x: x.mode().iat[0],
        'Latitude': 'median',
        'Longitude':'median',
    })
        
    # Заполнение, сбрасываем индекс на название, заполняем из справочника, восстанавливаем индекс
    df_filled = df.set_index(check_col).fillna(mapping).reset_index()    

    return visual_check, df_filled
        

In [13]:
visual_check, filled_train = check_column_for_fill(train, 'Field name')

In [14]:
filled_train.isna().sum().sum()

np.int64(136)

In [15]:
visual_check

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
218,BERYL,BERYL,UK,EUROPE,NORTH SEA NORTHERN,EXTENSION/EROSION,59.5498,1.5318,EXXONMOBIL,OFFSHORE,OIL,DECLINING PRODUCTION,RIFT,9200,JURASSIC,SANDSTONE,500.0,425.0,17.0,350.0
230,BERYL,LINNHE,UK,NaN,NaN,EXTENSION/EROSION,59.5498,1.5318,EXXONMOBIL,OFFSHORE,OIL,UNKNOWN,RIFT,9600,JURASSIC,SANDSTONE,300.0,120.0,14.0,100.0
139,CAROLINE,SWAN HILLS (BEAVERHILL LAKE A),NaN,NaN,NaN,COMPRESSION,NaN,NaN,NUMEROUS,ONSHORE,GAS-CONDENSATE,DECLINING PRODUCTION,FORELAND,11500,DEVONIAN,DOLOMITE,300.0,60.0,10.0,100.0
196,CAROLINE,CARDIUM (E POOL),CANADA,NORTH AMERICA,WESTERN CANADA,COMPRESSION,51.9278,-114.5405,NUMEROUS,ONSHORE,OIL,NEARLY DEPLETED,FORELAND,7975,CRETACEOUS,SANDSTONE,300.0,7.0,11.0,20.0
222,DURI,BEKASAP (KEDUA-PERTAMA SANDS),INDONESIA,FAR EAST,SUMATRA CENTRAL,INVERSION/STRIKE-SLIP/TRANSPRESSION/EXTENSION/...,1.2956,101.2304,CALTEX PACIFIC INDONESIA,ONSHORE,OIL,MATURE PRODUCTION,INVERSION/BACKARC,400,NEOGENE,SANDSTONE,280.0,150.0,34.0,1500.0
226,DURI,DURI (RINDU SAND),INDONESIA,NaN,NaN,INVERSION/STRIKE-SLIP/TRANSPRESSION/EXTENSION/...,1.2956,101.2304,CALTEX PACIFIC INDONESIA,ONSHORE,OIL,MATURE PRODUCTION,INVERSION/BACKARC,220,NEOGENE,SANDSTONE,120.0,52.0,32.0,1900.0
79,OROCUAL,SAN JUAN,NaN,NaN,NaN,COMPRESSION/EXTENSION/LINKED,NaN,NaN,PDVSA,ONSHORE,OIL,DECLINING PRODUCTION,THRUST/PASSIVE MARGIN,12550,CRETACEOUS-PALEOGENE,SANDSTONE,1000.0,650.0,6.0,5.0
144,OROCUAL,CARAPITA,VENEZUELA,LATIN AMERICA,EASTERN VENEZUELA,COMPRESSION/EXTENSION/LINKED,9.8667,-63.3333,PDVSA,ONSHORE,OIL,CONTINUING DEVELOPMENT,THRUST/FORELAND,10150,NEOGENE,SANDSTONE,300.0,160.0,13.0,35.0
49,YIBAL,KHUFF,NaN,NaN,NaN,COMPRESSION/EVAPORITE,NaN,NaN,PDO,ONSHORE,OIL,DECLINING PRODUCTION,SALT/FORELAND,9219,PERMIAN,DOLOMITE,670.0,250.0,15.0,5.0
173,YIBAL,SHUAIBA,OMAN,MIDDLE EAST,FAHUD SALT,COMPRESSION/EVAPORITE,22.1333,56.0000,PDO,ONSHORE,OIL,DECLINING PRODUCTION,SALT/FORELAND,4440,CRETACEOUS,CHALK,350.0,300.0,28.0,5.0


Видно, что можем смело заполнть пропуски по уже имеющимся данным.

In [16]:
# Проверим заполнение.

# как было
visual_check.head(2)

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
218,BERYL,BERYL,UK,EUROPE,NORTH SEA NORTHERN,EXTENSION/EROSION,59.5498,1.5318,EXXONMOBIL,OFFSHORE,OIL,DECLINING PRODUCTION,RIFT,9200,JURASSIC,SANDSTONE,500.0,425.0,17.0,350.0
230,BERYL,LINNHE,UK,NaN,NaN,EXTENSION/EROSION,59.5498,1.5318,EXXONMOBIL,OFFSHORE,OIL,UNKNOWN,RIFT,9600,JURASSIC,SANDSTONE,300.0,120.0,14.0,100.0


In [17]:
# вывод как заполнилось
filled_train.loc[[218, 230]]

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
218,BERYL,BERYL,UK,EUROPE,NORTH SEA NORTHERN,EXTENSION/EROSION,59.5498,1.5318,EXXONMOBIL,OFFSHORE,OIL,DECLINING PRODUCTION,RIFT,9200,JURASSIC,SANDSTONE,500.0,425.0,17.0,350.0
230,BERYL,LINNHE,UK,EUROPE,NORTH SEA NORTHERN,EXTENSION/EROSION,59.5498,1.5318,EXXONMOBIL,OFFSHORE,OIL,UNKNOWN,RIFT,9600,JURASSIC,SANDSTONE,300.0,120.0,14.0,100.0


In [18]:
# Применяем изменения
train = filled_train

In [19]:
train.isna().sum().sum()

np.int64(136)

Теперь попробуем определить по полю *Reservoir unit*

In [20]:
visual_check, filled_train = check_column_for_fill(train, 'Reservoir unit')

filled_train.isna().sum().sum()

np.int64(103)

In [21]:
visual_check

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
11,BADR EL DIN-2,BAHARIYA,NaN,NaN,NaN,EXTENSION,NaN,NaN,BAPETCO,ONSHORE,GAS-CONDENSATE,DECLINING PRODUCTION,RIFT,7546,CRETACEOUS,SANDSTONE,361.0,148.0,20.0,225.0
254,BADR EL DIN-1,BAHARIYA,EGYPT,AFRICA,ABU GHARADIG,EXTENSION,29.8589,28.5229,BAPETCO,ONSHORE,OIL,MATURE PRODUCTION,RIFT,11175,CRETACEOUS,LOW-RESISTIVITY SANDSTONE,115.0,46.0,12.0,15.0
12,BRIDGER LAKE,DAKOTA SANDSTONE (LOWER MEMBER),NaN,NaN,NaN,COMPRESSION/EROSION,NaN,NaN,BTA OIL PRODUCERS,ONSHORE,OIL,NEARLY DEPLETED,FORELAND,15460,CRETACEOUS,SANDSTONE,270.0,35.0,12.8,79.0
260,LUCKEY DITCH,DAKOTA SANDSTONE (LOWER MEMBER),USA,NORTH AMERICA,GREATER GREEN RIVER,COMPRESSION/EROSION,41.0204,-110.2755,WHITING OIL and GAS,ONSHORE,OIL,MATURE PRODUCTION,FORELAND,13250,CRETACEOUS,SANDSTONE,250.0,23.0,14.0,60.0
66,INDEFATIGABLE,LEMAN SANDSTONE,UK,NaN,NaN,INVERSION/COMPRESSION/EXTENSION/EVAPORITE/GRAVITY,53.3932,2.5239,SHELL /PERENCO,OFFSHORE,GAS,NEARLY DEPLETED,SUB-SALT/INVERSION,7400,PERMIAN,SANDSTONE,200.0,197.0,15.0,30.0
125,CLIPPER,LEMAN SANDSTONE,UK,EUROPE,NORTH SEA SOUTHERN,INVERSION/COMPRESSION/EXTENSION/EVAPORITE/GRAVITY,53.4682,1.7338,SHELL,OFFSHORE,GAS,DECLINING PRODUCTION,SUB-SALT/INVERSION,7420,PERMIAN,SANDSTONE,715.0,580.0,11.0,0.5
137,VICTOR,LEMAN SANDSTONE,UK,EUROPE,NORTH SEA SOUTHERN,INVERSION/COMPRESSION/EXTENSION/EVAPORITE/GRAVITY,53.3299,2.3626,CONOCOPHILLIPS,OFFSHORE,GAS,DECLINING PRODUCTION,SUB-SALT/INVERSION,8175,PERMIAN,SANDSTONE,375.0,369.0,16.0,52.0
232,BARQUE,LEMAN SANDSTONE,UK,EUROPE,NORTH SEA SOUTHERN,INVERSION/COMPRESSION/EXTENSION/EVAPORITE/GRAVITY,53.6204,NaN,SHELL,OFFSHORE,GAS,DECLINING PRODUCTION,SUB-SALT/INVERSION,6900,PERMIAN,SANDSTONE,750.0,570.0,11.0,0.5
154,ELKHORN RANCH,MISSION CANYON (MADISON),USA,NORTH AMERICA,WILLISTON,COMPRESSION,47.2376,-103.5028,TENNECO /CENEX /SHELL,ONSHORE,OIL,DECLINING PRODUCTION,INTRACRATONIC,9125,CARBONIFEROUS,DOLOMITE,105.0,40.0,16.0,25.0
155,BEAVER LODGE,MISSION CANYON (MADISON),NaN,NaN,NaN,COMPRESSION,NaN,NaN,AMERADA HESS,ONSHORE,OIL,REJUVENATING,INTRACRATONIC,8350,CARBONIFEROUS,LIMESTONE,120.0,20.0,6.0,1.0


In [22]:
# Проверим заполнение.

# как было
indices = visual_check.tail(3).index
display(visual_check.tail(3))
# как заполнилось
filled_train.loc[indices]

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
99,KARACHAGANAK,UNNAMED,KAZAKHSTAN,FORMER SOVIET UNION,CASPIAN NORTH,COMPRESSION/EVAPORITE,51.3158,53.2600,KARACHAGANAK INTEGRATED ORAGANIZATION,ONSHORE,GAS-CONDENSATE,REJUVENATING,SUB-SALT/FORELAND,11870,DEVONIAN-PERMIAN,LIMESTONE,6600.0,2600.0,10.0,8.0
185,PARENTIS,UNNAMED,FRANCE,EUROPE,AQUITAINE,COMPRESSION/EROSION,44.3367,-1.0959,VERMILION,ONSHORE-OFFSHORE,OIL,NEARLY DEPLETED,FORELAND,6550,CRETACEOUS,DOLOMITE,1300.0,200.0,11.0,10.0
192,ORENBURG,UNNAMED,NaN,NaN,NaN,COMPRESSION/EVAPORITE,NaN,NaN,ORENBURGGAZPROM,ONSHORE,GAS,UNKNOWN,SUB-SALT/FORELAND,4232,CARBONIFEROUS-PERMIAN,LIMESTONE,1360.0,563.0,11.0,2.0


,Reservoir unit,Field name,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
99,UNNAMED,KARACHAGANAK,KAZAKHSTAN,FORMER SOVIET UNION,CASPIAN NORTH,COMPRESSION/EVAPORITE,51.31580,53.26000,KARACHAGANAK INTEGRATED ORAGANIZATION,ONSHORE,GAS-CONDENSATE,REJUVENATING,SUB-SALT/FORELAND,11870,DEVONIAN-PERMIAN,LIMESTONE,6600.0,2600.0,10.0,8.0
185,UNNAMED,PARENTIS,FRANCE,EUROPE,AQUITAINE,COMPRESSION/EROSION,44.33670,-1.09590,VERMILION,ONSHORE-OFFSHORE,OIL,NEARLY DEPLETED,FORELAND,6550,CRETACEOUS,DOLOMITE,1300.0,200.0,11.0,10.0
192,UNNAMED,ORENBURG,FRANCE,EUROPE,AQUITAINE,COMPRESSION/EVAPORITE,47.82625,26.08205,ORENBURGGAZPROM,ONSHORE,GAS,UNKNOWN,SUB-SALT/FORELAND,4232,CARBONIFEROUS-PERMIAN,LIMESTONE,1360.0,563.0,11.0,2.0


Есть поле *UNNAMED*, по которому восстановление не получится. Проверим есть ли у этого оператора другие месторождения.

In [23]:
train[train['Operator company'] == 'ORENBURGGAZPROM']

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
192,ORENBURG,UNNAMED,NaN,NaN,NaN,COMPRESSION/EVAPORITE,NaN,NaN,ORENBURGGAZPROM,ONSHORE,GAS,UNKNOWN,SUB-SALT/FORELAND,4232,CARBONIFEROUS-PERMIAN,LIMESTONE,1360.0,563.0,11.0,2.0


Заменим это поле на уникальное

In [24]:
mask = train['Reservoir unit'] == 'UNNAMED'
train.loc[mask, 'Reservoir unit'] = train.loc[mask, 'Field name'] + ' UNNAMED'


Сделаем ещё раз

In [25]:
visual_check, filled_train = check_column_for_fill(train, 'Reservoir unit')

filled_train.isna().sum().sum()

np.int64(108)

In [26]:
# Проверим заполнение.

# как было
indices = visual_check.head(2).index
display(visual_check.head(2))
# как заполнилось
filled_train.loc[indices]

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
11,BADR EL DIN-2,BAHARIYA,NaN,NaN,NaN,EXTENSION,NaN,NaN,BAPETCO,ONSHORE,GAS-CONDENSATE,DECLINING PRODUCTION,RIFT,7546,CRETACEOUS,SANDSTONE,361.0,148.0,20.0,225.0
254,BADR EL DIN-1,BAHARIYA,EGYPT,AFRICA,ABU GHARADIG,EXTENSION,29.8589,28.5229,BAPETCO,ONSHORE,OIL,MATURE PRODUCTION,RIFT,11175,CRETACEOUS,LOW-RESISTIVITY SANDSTONE,115.0,46.0,12.0,15.0


,Reservoir unit,Field name,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
11,BAHARIYA,BADR EL DIN-2,EGYPT,AFRICA,ABU GHARADIG,EXTENSION,29.8589,28.5229,BAPETCO,ONSHORE,GAS-CONDENSATE,DECLINING PRODUCTION,RIFT,7546,CRETACEOUS,SANDSTONE,361.0,148.0,20.0,225.0
254,BAHARIYA,BADR EL DIN-1,EGYPT,AFRICA,ABU GHARADIG,EXTENSION,29.8589,28.5229,BAPETCO,ONSHORE,OIL,MATURE PRODUCTION,RIFT,11175,CRETACEOUS,LOW-RESISTIVITY SANDSTONE,115.0,46.0,12.0,15.0


In [27]:
# Применяем изменения
train = filled_train

train.isna().sum().sum()

np.int64(108)

In [323]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

Country       18
Region        26
Basin name    26
Latitude      18
Longitude     20
dtype: int64

In [324]:
train[train['Country'].notna() & train['Region'].isna()]

,Reservoir unit,Field name,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
16,BASSEIN (ZONE B),HEERA,INDIA,NaN,NaN,EXTENSION,18.1720,72.3130,ONGC,OFFSHORE,OIL,DECLINING PRODUCTION,PASSIVE MARGIN,4542,PALEOGENE,LIMESTONE,230.0,82.0,12.0,0.8
21,HUTTON,GIDGEALPA,AUSTRALIA,NaN,NaN,INVERSION/COMPRESSION/EXTENSION,-28.0000,140.0000,SANTOS,ONSHORE,OIL,PLATEAU PRODUCTION,INTRACRATONIC,5392,JURASSIC,SANDSTONE,600.0,50.0,18.0,1000.0
40,YATES,NORTH WARD-ESTES,USA,NaN,NaN,COMPRESSION,31.5777,-102.9918,WHITING OIL and GAS,ONSHORE,OIL,REJUVENATING,FORELAND,2600,PERMIAN,SILTSTONE,300.0,100.0,11.0,16.0
82,F2-F0,WEST TEBUK,RUSSIA,NaN,NaN,COMPRESSION/EVAPORITE,63.6375,54.9269,KOMINEFT,ONSHORE,OIL,DECLINING PRODUCTION,INVERSION/FORELAND,3493,DEVONIAN,LIMESTONE,980.0,200.0,12.0,200.0
107,LOWER GANCHAIGOU,GASIKULE,CHINA,NaN,NaN,COMPRESSION,38.1000,90.8700,PETROCHINA,ONSHORE,OIL,PLATEAU PRODUCTION,THRUST,10427,PALEOGENE,SANDSTONE,804.0,93.0,14.0,48.0
143,TEMBLOR,COALINGA,USA,NaN,NaN,COMPRESSION/STRIKE-SLIP/TRANSPRESSION/BASEMENT-I,36.2674,-120.3659,CHEVRON AND OTHERS,ONSHORE,OIL,MATURE PRODUCTION,WRENCH/FOREARC,500,NEOGENE,SHALY SANDSTONE,700.0,190.0,34.0,850.0
214,GRAYBURG,SOUTH COWDEN,USA,NaN,NaN,COMPRESSION,31.7490,-102.4430,NUMEROUS,ONSHORE,OIL,DECLINING PRODUCTION,FORELAND,4100,PERMIAN,DOLOMITE,400.0,90.0,8.8,15.0
229,POKUR (PK1-6),URENGOY,RUSSIA,NaN,NaN,COMPRESSION,66.0533,76.9497,TYUMENNEFTEGAZ,ONSHORE,GAS,DECLINING PRODUCTION,INTRACRATONIC,3514,CRETACEOUS,SANDSTONE,700.0,440.0,30.0,330.0


In [268]:
# Разделим данные на две таблицы, где есть пропуски и где все поля заполнены

# .any(axis='columns') - перебирает все строки и помечает всю строку True, если есть хотя бы один NaN в любой колонке
missed = train[train.isna().any(axis='columns')] 

filled = train.dropna()

In [269]:
# Получим список заполненных полей
col_filled = 'Operator company'
filled_names = filled[col_filled].unique()

# Отберём те строки, в таблице с пропусками, которые есть без пропусков
common_names = missed[missed[col_filled].isin(filled_names)]

In [270]:
# Выведем результат из общей таблицы, чтобы визауально оценить возможность заполнения пропусков
train[train[col_filled].isin(common_names[col_filled])].sort_values(col_filled)

,Reservoir unit,Field name,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
37,ULA,ULA,NaN,NaN,NaN,INVERSION/COMPRESSION/EXTENSION/EVAPORITE,NaN,NaN,BP,OFFSHORE,OIL,MATURE PRODUCTION,SALT/INVERSION/RIFT,10744,JURASSIC,SANDSTONE,410.0,377.0,16.5,300.00
38,WATT MOUNTAIN (GILWOOD A POOL),NIPISI,CANADA,NORTH AMERICA,WESTERN CANADA,COMPRESSION,55.8199,-115.1317,BP,ONSHORE,OIL,MATURE PRODUCTION,FORELAND,5500,DEVONIAN,SANDSTONE,100.0,14.0,13.5,250.00
64,SHERWOOD,WYTCH FARM,UK,EUROPE,WESSEX,INVERSION/COMPRESSION/EXTENSION,50.6672,-2.0278,BP,ONSHORE-OFFSHORE,OIL,DECLINING PRODUCTION,RIFT/INVERSION,4954,TRIASSIC,SANDSTONE,525.0,150.0,18.0,100.00
91,FORTIES,EVEREST,UK,EUROPE,NORTH SEA CENTRAL,EXTENSION/EVAPORITE/GRAVITY,57.7500,1.8100,BP,OFFSHORE,GAS-CONDENSATE,DECLINING PRODUCTION,RIFT,8008,PALEOGENE,SANDSTONE,85.0,50.0,22.0,27.00
93,LANSING-KANSAS CITY,VICTORY,USA,NORTH AMERICA,ANADARKO,COMPRESSION/EROSION,37.3971,-100.9544,BP,ONSHORE,OIL,MATURE PRODUCTION,FORELAND,4075,CARBONIFEROUS,LIMESTONE,150.0,10.0,22.0,85.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,UNNAMED (FENGHUADIAN BLOCK),ZAOYUAN,CHINA,FAR EAST,BOHAI,EXTENSION,38.2100,117.0600,PETROCHINA,ONSHORE,OIL,MATURE PRODUCTION,RIFT,9843,MESOZOIC,VOLCANICS,492.0,194.0,12.7,3.59
100,CLEARFORK-GLORIETA,NORTH ROBERTSON,NaN,NaN,NaN,COMPRESSION,NaN,NaN,TOTAL,ONSHORE,OIL,SECOND PLATEAU PRODUTION,FORELAND,5800,PERMIAN,DOLOMITE,1300.0,400.0,4.0,0.40
169,STATFJORD,ALWYN NORTH,UK,EUROPE,NORTH SEA NORTHERN,EXTENSION/EROSION,60.7833,1.7333,TOTAL,OFFSHORE,GAS-CONDENSATE,MATURE PRODUCTION,RIFT,10545,TRIASSIC-JURASSIC,SANDSTONE,869.0,512.0,13.5,330.00
40,YATES,NORTH WARD-ESTES,USA,NaN,NaN,COMPRESSION,31.5777,-102.9918,WHITING OIL and GAS,ONSHORE,OIL,REJUVENATING,FORELAND,2600,PERMIAN,SILTSTONE,300.0,100.0,11.0,16.00


In [272]:
col_replace_name = col_filled

In [273]:
# создадим таблицу справочник с модой и медианой по названию месторождения
col_replace_name_map = filled.groupby(col_replace_name).agg({
    'Country': lambda x: x.mode().iat[0], # так как мода вычисляется, то для указания индекса используем iat 
    'Region': lambda x: x.mode().iat[0],
    'Basin name': lambda x: x.mode().iat[0],
    'Latitude': 'median',
    'Longitude':'median',
})

In [274]:
col_replace_name_map.head(5)

,Country,Region,Basin name,Latitude,Longitude
Operator company,,,,,
ADCO,UAE,MIDDLE EAST,RUB AL KHALI,23.41450,53.75845
ADMA /ZADCO,UAE,MIDDLE EAST,RUB AL KHALI,24.86670,53.68330
AFGHANGAS,AFGHANISTAN,FAR EAST,AMU DARYA,36.61670,65.70000
ALBERTA ENERGY COMPANY /WESTCOAST PETROLEUM,CANADA,NORTH AMERICA,WESTERN CANADA,50.28510,-111.14270
AMERADA HESS,USA,NORTH AMERICA,WILLISTON,48.80115,-100.96630


In [275]:
# Заполнение, сбрасываем индекс на название, заполняем из справочника, восстанавливаем индекс
train = train.set_index(col_replace_name).fillna(col_replace_name_map).reset_index()
test = test.set_index(col_replace_name).fillna(col_replace_name_map).reset_index()

In [276]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

Country        8
Region        11
Basin name    11
Latitude       8
Longitude     10
dtype: int64

In [277]:
train[train['Country'].isna()]

,Operator company,Reservoir unit,Field name,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
15,NEXEN,SGIATH-PIPER,SCOTT,NaN,NaN,NaN,EXTENSION/EVAPORITE,NaN,NaN,OFFSHORE,OIL,DECLINING PRODUCTION,RIFT,9940,JURASSIC,SANDSTONE,360.0,290.0,16.0,100.00
35,MANGYSTAUMUNAIGAZ,UNITS XIII-XVIII,UZEN,NaN,NaN,NaN,INVERSION/COMPRESSION/EXTENSION,NaN,NaN,ONSHORE,OIL,MATURE PRODUCTION,INVERSION/RIFT,3440,JURASSIC,SANDSTONE,920.0,436.0,21.0,235.00
50,MAGELLAN PETROLEUM,PACOOTA AND STAIRWAY,PALM VALLEY,NaN,NaN,NaN,COMPRESSION/EVAPORITE,NaN,NaN,ONSHORE,GAS,DECLINING PRODUCTION,INTRACRATONIC,5656,ORDOVICIAN,SANDSTONE,2300.0,174.0,4.0,0.01
80,ANSCHUTZ AND OTHERS,MISSION CANYON (FROBISHER-ALIDA),GLENBURN,NaN,NaN,NaN,COMPRESSION,NaN,NaN,ONSHORE,OIL,MATURE PRODUCTION,INTRACRATONIC,4840,CARBONIFEROUS,LIMESTONE,100.0,10.0,17.0,24.00
102,CANADIAN SUPERIOR,UPPER MANNVILE (A POOL),TABER NORTH,NaN,NaN,NaN,COMPRESSION/EROSION,NaN,NaN,ONSHORE,OIL,MATURE PRODUCTION,FORELAND,3050,CRETACEOUS,SANDSTONE,165.0,35.0,25.0,2000.00
168,WILDFIRE PARTNERS INC,UPPER MINNELUSA (C SAND),ROURKE GAP,NaN,NaN,NaN,COMPRESSION,NaN,NaN,ONSHORE,OIL,NEARLY DEPLETED,FORELAND,10225,PERMIAN,SANDSTONE,100.0,25.0,11.3,97.00
186,PLUSPETROL,VIIVIAN,CASHIRIARI,NaN,NaN,NaN,COMPRESSION,NaN,NaN,ONSHORE,GAS-CONDENSATE,UNDEVELOPED,THRUST,6150,CRETACEOUS,SANDSTONE,245.0,210.0,13.0,1000.00
192,ORENBURGGAZPROM,ORENBURG UNNAMED,ORENBURG,NaN,NaN,NaN,COMPRESSION/EVAPORITE,NaN,NaN,ONSHORE,GAS,UNKNOWN,SUB-SALT/FORELAND,4232,CARBONIFEROUS-PERMIAN,LIMESTONE,1360.0,563.0,11.0,2.00


In [278]:
# Разделим данные на две таблицы, где есть пропуски и где все поля заполнены

# .any(axis='columns') - перебирает все строки и помечает всю строку True, если есть хотя бы один NaN в любой колонке
missed = train[train.isna().any(axis='columns')] 

filled = train.dropna()

In [280]:
# Получим список заполненных полей
col_filled = 'Structural setting'
filled_names = filled[col_filled].unique()

# Отберём те строки, в таблице с пропусками, которые есть без пропусков
common_names = missed[missed[col_filled].isin(filled_names)]

In [282]:
# Выведем результат из общей таблицы, чтобы визауально оценить возможность заполнения пропусков
train[train[col_filled].isin(common_names[col_filled])].sort_values(col_filled)

,Operator company,Reservoir unit,Field name,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
0,NIZHNEVOLZHSKNET,MELEKESKIAN,ZHIRNOV,RUSSIA,FORMER SOVIET UNION,VOLGA-URAL,COMPRESSION/EVAPORITE,51.0000,44.8042,ONSHORE,OIL,DECLINING PRODUCTION,FORELAND,1870,CARBONIFEROUS,SANDSTONE,262.0,33.0,24.0,30.0
2,SAUDI ARAMCO,ARAB D,ABQAIQ,SAUDI ARABIA,MIDDLE EAST,THE GULF,COMPRESSION/EVAPORITE,26.0800,49.8100,ONSHORE,OIL,REJUVENATING,FORELAND,6050,JURASSIC,LIMESTONE,250.0,184.0,21.0,410.0
4,NUMEROUS,NISKU (PEMBINA L POOL),WEST PEMBINA,CANADA,NORTH AMERICA,WESTERN CANADA,COMPRESSION,53.2287,-115.8008,ONSHORE,OIL,UNKNOWN,FORELAND,9306,DEVONIAN,DOLOMITE,233.0,167.0,11.8,1407.0
8,NUMEROUS,CLEVELAND,ELLIS RANCH,USA,NORTH AMERICA,ANADARKO,COMPRESSION/EROSION,36.2724,-100.7161,ONSHORE,GAS,DECLINING PRODUCTION,FORELAND,6500,CARBONIFEROUS,SHALY SANDSTONE,150.0,26.0,14.0,0.4
9,BANF AND AQUITAINE,LEDUC,STRACHAN,CANADA,NORTH AMERICA,WESTERN CANADA,COMPRESSION,52.4500,-115.4286,ONSHORE,GAS,NEARLY DEPLETED,FORELAND,12068,DEVONIAN,DOLOMITE,900.0,535.0,7.8,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,PETROCHINA,BAIYANGHE (JIANQUANZI L SAND),LAOJUNMIAO,CHINA,FAR EAST,JIUQUAN,COMPRESSION/EROSION/EXTENSION/LINKED,39.7658,97.5663,ONSHORE,OIL,NEARLY DEPLETED,THRUST,1362,PALEOGENE,SANDSTONE,180.0,39.0,23.0,619.0
216,PETROCHINA,BAIYANGHE (JIANQUANZI M SAND),LAOJUNMIAO,CHINA,FAR EAST,JIUQUAN,COMPRESSION/EROSION/EXTENSION/LINKED,39.7658,97.5663,ONSHORE,OIL,MATURE PRODUCTION,THRUST,1532,PALEOGENE,SANDSTONE,210.0,82.0,17.8,24.0
224,CHEVRON,HUNTON (CHIMNEY HILL-HENRYHOUSE),MILLS RANCH,USA,NORTH AMERICA,ANADARKO,COMPRESSION/EROSION,35.3676,-100.0793,ONSHORE,GAS,NEARLY DEPLETED,THRUST,19888,SILURIAN,DOLOMITE,930.0,94.0,6.0,7.0
277,PETROCHINA,KELAMAYI,KELAMAYI,CHINA,FAR EAST,JUNGGAR (ZHUNGEER),COMPRESSION/EROSION,45.5417,84.8917,ONSHORE,OIL,DECLINING PRODUCTION,THRUST,6100,TRIASSIC,CONGLOMERATE,750.0,43.0,18.0,279.0


In [283]:
col_replace_name = col_filled

In [284]:
# создадим таблицу справочник с модой и медианой по названию месторождения
col_replace_name_map = filled.groupby(col_replace_name).agg({
    'Country': lambda x: x.mode().iat[0], # так как мода вычисляется, то для указания индекса используем iat 
    'Region': lambda x: x.mode().iat[0],
    'Basin name': lambda x: x.mode().iat[0],
    'Latitude': 'median',
    'Longitude':'median',
})

In [285]:
col_replace_name_map.head(5)

,Country,Region,Basin name,Latitude,Longitude
Structural setting,,,,,
BACKARC,INDONESIA,FAR EAST,JAVA NORTHWEST,-0.1670,109.7833
DELTA/FORELAND,MALAYSIA,FAR EAST,SABAH,5.6284,114.8906
DELTA/PASSIVE MARGIN,USA,NORTH AMERICA,GULF OF MEXICO NORTHERN ONSHORE,26.5307,-90.9141
DELTA/SALT/PASSIVE MARGIN,USA,NORTH AMERICA,GULF OF MEXICO NORTHERN ONSHORE,29.8800,-94.0000
DELTA/WRENCH,MALAYSIA,FAR EAST,SABAH,4.7050,113.6940


In [286]:
# Заполнение, сбрасываем индекс на название, заполняем из справочника, восстанавливаем индекс
train = train.set_index(col_replace_name).fillna(col_replace_name_map).reset_index()
test = test.set_index(col_replace_name).fillna(col_replace_name_map).reset_index()

In [287]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

Region        1
Basin name    1
dtype: int64

In [237]:
train[train['Country'] == 'RUSSIA']

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
0,ZHIRNOV,MELEKESKIAN,RUSSIA,FORMER SOVIET UNION,VOLGA-URAL,COMPRESSION/EVAPORITE,51.0000,44.8042,NIZHNEVOLZHSKNET,ONSHORE,OIL,DECLINING PRODUCTION,FORELAND,1870,CARBONIFEROUS,SANDSTONE,262.0,33.0,24.0,30.0
32,ROMASHKINO,PASHIY (D-1 POOL),RUSSIA,FORMER SOVIET UNION,VOLGA-URAL,COMPRESSION/EROSION,55.3433,52.8447,TATNEFT,ONSHORE,OIL,MATURE PRODUCTION,FORELAND,5463,DEVONIAN,SANDSTONE,115.0,29.0,19.0,375.0
57,MURAVLENKOV,MEGION,RUSSIA,FORMER SOVIET UNION,SIBERIAN WESTERN,COMPRESSION,63.9900,74.9500,SIBNEFT (MURAVLENKOVSKNEFT),ONSHORE,OIL,DECLINING PRODUCTION,INTRACRATONIC,8399,CRETACEOUS,SANDSTONE,1476.0,102.0,18.5,45.0
61,POKACHEV,VARTOV (BV6),RUSSIA,FORMER SOVIET UNION,SIBERIAN WESTERN,COMPRESSION,61.5830,75.1833,SURGUTNEFTEGAS,ONSHORE,OIL,DECLINING PRODUCTION,INTRACRATONIC,7529,CRETACEOUS,SANDSTONE,52.0,19.0,20.0,170.0
75,KULESHOV,A3,RUSSIA,FORMER SOVIET UNION,VOLGA-URAL,COMPRESSION/EVAPORITE,52.8381,51.2136,KUYBYSHEVNEFT,ONSHORE,OIL,NEARLY DEPLETED,FORELAND,5266,CARBONIFEROUS,SANDSTONE,70.0,46.0,20.0,177.0
82,WEST TEBUK,F2-F0,RUSSIA,NaN,NaN,COMPRESSION/EVAPORITE,63.6375,54.9269,KOMINEFT,ONSHORE,OIL,DECLINING PRODUCTION,INVERSION/FORELAND,3493,DEVONIAN,LIMESTONE,980.0,200.0,12.0,200.0
142,VERKHNECHONA,NEPA (VCH1-VCH2),RUSSIA,FORMER SOVIET UNION,SIBERIAN EASTERN,INVERSION/COMPRESSION/EXTENSION,60.8929,109.0385,VERKHNECHONSKNEFTEGAZ,ONSHORE,OIL,DEVELOPING,INVERSION/RIFT,5216,PROTEROZOIC,SANDSTONE,200.0,25.0,10.0,209.0
165,VAREGAN,MEGION-VAREGAN (A5-B10),RUSSIA,FORMER SOVIET UNION,SIBERIAN WESTERN,COMPRESSION,62.6167,77.6622,NOYABRSKNEFTEGAS,ONSHORE,OIL,DECLINING PRODUCTION,INTRACRATONIC,5577,CRETACEOUS,SANDSTONE,3150.0,429.0,23.6,187.0
171,YAREGA,KOYVA,RUSSIA,FORMER SOVIET UNION,TIMAN-PECHORA,COMPRESSION/EVAPORITE,63.2992,53.6303,KOMINEFT,ONSHORE,OIL,PLATEAU PRODUCTION,INVERSION/FORELAND,480,DEVONIAN,SANDSTONE,100.0,56.0,21.0,1000.0
177,KOVYKTA,PARFENOV,RUSSIA,FORMER SOVIET UNION,SIBERIAN EASTERN,COMPRESSION/EVAPORITE,55.4167,106.0000,RUSIA PETROLEUM,ONSHORE,GAS,UNDEVELOPED,INTRACRATONIC,4000,PROTEROZOIC,SANDSTONE,200.0,49.0,16.0,5.0


Теперь попробуем определить по полю *Operator company*

In [233]:
# Разделим данные на две таблицы, где есть пропуски и где все поля заполнены

# .any(axis='columns') - перебирает все строки и помечает всю строку True, если есть хотя бы один NaN в любой колонке
missed = train[train.isna().any(axis='columns')] 

filled = train.dropna()

In [234]:
# Получим список заполненных полей
col_filled = 'Operator company'
filled_names = filled[col_filled].unique()

# Отберём те строки, в таблице с пропусками, которые есть без пропусков
common_names = missed[missed[col_filled].isin(filled_names)]

In [235]:
# Выведем результат из общей таблицы, чтобы визауально оценить возможность заполнения пропусков
train[train[col_filled].isin(common_names[col_filled])].sort_values(col_filled)

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
6,WESTHOPE SOUTH,CHARLES,USA,NORTH AMERICA,WILLISTON,COMPRESSION,48.8521,-101.0130,AMERADA HESS,ONSHORE,OIL,MATURE PRODUCTION,INTRACRATONIC,3275,CARBONIFEROUS,DOLOMITIC LIMESTONE,43.0,15.5,10.0,2.6
14,ANGUS,ANGUS SAND,UK,EUROPE,NORTH SEA CENTRAL,EXTENSION/EROSION,56.1648,3.0593,AMERADA HESS,OFFSHORE,OIL,DECLINING PRODUCTION,RIFT,10130,JURASSIC,SANDSTONE,100.0,60.0,18.0,3000.0
155,BEAVER LODGE,MISSION CANYON (MADISON),NaN,NaN,NaN,COMPRESSION,NaN,NaN,AMERADA HESS,ONSHORE,OIL,REJUVENATING,INTRACRATONIC,8350,CARBONIFEROUS,LIMESTONE,120.0,20.0,6.0,1.0
162,BALDPATE,BIG SAND,USA,NORTH AMERICA,GULF OF MEXICO NORTHERN OFFSHORE,GRAVITY/EXTENSION/EVAPORITE/SYNSEDIMENTATION,27.6900,-91.7300,AMERADA HESS,OFFSHORE,GAS,PLATEAU PRODUCTION,SALT/PASSIVE MARGIN,13850,NEOGENE,SANDSTONE,118.0,100.0,27.0,836.0
203,NEWBURG,CHARLES,USA,NORTH AMERICA,WILLISTON,COMPRESSION,48.7502,-100.9196,AMERADA HESS,ONSHORE,OIL,MATURE PRODUCTION,INTRACRATONIC,3310,CARBONIFEROUS,DOLOMITIC LIMESTONE,120.0,17.8,10.0,2.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,CABIN CREEK,RED RIVER,USA,NORTH AMERICA,WILLISTON,COMPRESSION,46.5782,-104.4303,SHELL,ONSHORE,OIL,NEARLY DEPLETED,INTRACRATONIC,8750,ORDOVICIAN,DOLOMITE,500.0,40.0,13.0,8.0
100,NORTH ROBERTSON,CLEARFORK-GLORIETA,NaN,NaN,NaN,COMPRESSION,NaN,NaN,TOTAL,ONSHORE,OIL,SECOND PLATEAU PRODUTION,FORELAND,5800,PERMIAN,DOLOMITE,1300.0,400.0,4.0,0.4
169,ALWYN NORTH,STATFJORD,UK,EUROPE,NORTH SEA NORTHERN,EXTENSION/EROSION,60.7833,1.7333,TOTAL,OFFSHORE,GAS-CONDENSATE,MATURE PRODUCTION,RIFT,10545,TRIASSIC-JURASSIC,SANDSTONE,869.0,512.0,13.5,330.0
40,NORTH WARD-ESTES,YATES,USA,NaN,NaN,COMPRESSION,31.5777,-102.9918,WHITING OIL and GAS,ONSHORE,OIL,REJUVENATING,FORELAND,2600,PERMIAN,SILTSTONE,300.0,100.0,11.0,16.0


In [236]:
common_names

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
11,BADR EL DIN-2,BAHARIYA,NaN,NaN,NaN,EXTENSION,NaN,NaN,BAPETCO,ONSHORE,GAS-CONDENSATE,DECLINING PRODUCTION,RIFT,7546,CRETACEOUS,SANDSTONE,361.0,148.0,20.0,225.00
16,HEERA,BASSEIN (ZONE B),INDIA,NaN,NaN,EXTENSION,18.1720,72.3130,ONGC,OFFSHORE,OIL,DECLINING PRODUCTION,PASSIVE MARGIN,4542,PALEOGENE,LIMESTONE,230.0,82.0,12.0,0.80
33,GASIKULE,UPPER GANCHAIGOU-LOWER YOUSHAS,NaN,NaN,NaN,COMPRESSION,NaN,NaN,PETROCHINA,ONSHORE,OIL,PLATEAU PRODUCTION,THRUST,4199,NEOGENE,SANDSTONE,3281.0,81.0,19.0,102.00
37,ULA,ULA,NaN,NaN,NaN,INVERSION/COMPRESSION/EXTENSION/EVAPORITE,NaN,NaN,BP,OFFSHORE,OIL,MATURE PRODUCTION,SALT/INVERSION/RIFT,10744,JURASSIC,SANDSTONE,410.0,377.0,16.5,300.00
40,NORTH WARD-ESTES,YATES,USA,NaN,NaN,COMPRESSION,31.5777,-102.9918,WHITING OIL and GAS,ONSHORE,OIL,REJUVENATING,FORELAND,2600,PERMIAN,SILTSTONE,300.0,100.0,11.0,16.00
55,WUBAITI,HUANGLONG,NaN,NaN,NaN,COMPRESSION,NaN,NaN,PETROCHINA,ONSHORE,GAS,DEVELOPING,THRUST,13780,CARBONIFEROUS,DOLOMITE,98.0,79.0,6.0,0.77
82,WEST TEBUK,F2-F0,RUSSIA,NaN,NaN,COMPRESSION/EVAPORITE,63.6375,54.9269,KOMINEFT,ONSHORE,OIL,DECLINING PRODUCTION,INVERSION/FORELAND,3493,DEVONIAN,LIMESTONE,980.0,200.0,12.0,200.00
89,HARMATTAN-ELKTON,TURNER VALLEY (RUNDLE C POOL),NaN,NaN,NaN,COMPRESSION/EROSION,NaN,NaN,BP AND OTHERS,ONSHORE,GAS-CONDENSATE,MATURE PRODUCTION,FORELAND,8750,CARBONIFEROUS,DOLOMITE,140.0,70.0,12.0,125.00
92,TIA JUANA,LA ROSA-LAGUNILLAS (TIA JUANA ONSHORE AREA),NaN,NaN,NaN,COMPRESSION/EROSION,NaN,NaN,PDVSA,ONSHORE-OFFSHORE,OIL,MATURE PRODUCTION,WRENCH/FORELAND,1400,NEOGENE,SANDSTONE,500.0,120.0,35.0,1200.00
98,EMPIRE ABO,ABO,NaN,NaN,NaN,COMPRESSION,NaN,NaN,BP,ONSHORE,OIL,NEARLY DEPLETED,FORELAND,5600,PERMIAN,DOLOMITE,300.0,151.0,10.0,50.00


In [220]:
train[train['Operator company'] == 'BP']['Country'].value_counts()

Country
USA          4
UK           3
CANADA       1
INDONESIA    1
Name: count, dtype: int64

In [221]:
train['Structural setting'].value_counts()

Structural setting
FORELAND                      78
RIFT                          46
INTRACRATONIC                 33
THRUST                        18
PASSIVE MARGIN                13
SALT/FORELAND                 13
SALT/PASSIVE MARGIN           10
INVERSION/RIFT                 9
INVERSION/BACKARC              8
DELTA/PASSIVE MARGIN           7
SALT/INVERSION/RIFT            6
DELTA/SALT/PASSIVE MARGIN      5
SUB-SALT/FORELAND              4
THRUST/FORELAND                4
SUB-SALT/INVERSION             4
RIFT/SALT                      4
SUB-THRUST/FORELAND            4
SALT/RIFT                      3
WRENCH                         3
RIFT/INVERSION                 3
BACKARC                        3
SUB-SALT/RIFT                  3
THRUST/SUB-THRUST/FORELAND     2
INVERSION/FORELAND             2
WRENCH/FORELAND                2
SUB-THRUST                     2
FORELAND/SALT                  2
FORELAND/THRUST                2
WRENCH/DELTA                   2
WRENCH/INVERSION/BACKARC

In [222]:
train['Hydrocarbon type'].value_counts()

Hydrocarbon type
OIL               231
GAS                46
GAS-CONDENSATE     30
CARBON DIOXIDE      2
Name: count, dtype: int64

In [223]:
train['Field name'].value_counts()

Field name
ZAKUM                3
ERSKINE              3
LAOJUNMIAO           3
WESTHOPE SOUTH       2
GASIKULE             2
                    ..
HIDES                1
DRAKE POINT          1
ALTAMONT-BLUEBELL    1
BELL CREEK           1
WELL DRAW            1
Name: count, Length: 285, dtype: int64

In [224]:
train['Reservoir status'].value_counts()

Reservoir status
DECLINING PRODUCTION        88
MATURE PRODUCTION           63
NEARLY DEPLETED             55
PLATEAU PRODUCTION          30
DEVELOPING                  22
REJUVENATING                18
UNKNOWN                      9
UNDEVELOPED                  7
ABANDONED                    6
SECOND PLATEAU PRODUTION     6
CONTINUING DEVELOPMENT       3
TEMPORARILY SHUT-IN          1
DEPLETED                     1
Name: count, dtype: int64

In [225]:
train['Reservoir unit'].value_counts()

Reservoir unit
BRENT                             8
SAN ANDRES                        7
SHUAIBA                           5
LEMAN SANDSTONE                   4
TOR-EKOFISK                       4
                                 ..
SPEARFISH                         1
IMBURU-TORO                       1
DRAKE POINT-INTREPID INLET        1
GREEN RIVER AND COLTON/WASATCH    1
MESAVERDE (TEAPOT SAND)           1
Name: count, Length: 258, dtype: int64

In [226]:
train['Operator company'].value_counts()

Operator company
NUMEROUS                        28
PETROCHINA                      17
CHEVRON                         16
BP                              12
SHELL                           10
                                ..
EXXONMOBIL /SMNG                 1
OIL SEARCH LTD                   1
PANARCTIC OILS                   1
SAMUEL GARY                      1
MATRIX PRODUCTION AND OTHERS     1
Name: count, Length: 138, dtype: int64

In [227]:
train['Region'].value_counts()

Region
NORTH AMERICA          111
FAR EAST                47
EUROPE                  40
FORMER SOVIET UNION     25
MIDDLE EAST             22
AFRICA                  19
LATIN AMERICA           13
Name: count, dtype: int64

In [228]:
train['Structural setting'].value_counts()

Structural setting
FORELAND                      78
RIFT                          46
INTRACRATONIC                 33
THRUST                        18
PASSIVE MARGIN                13
SALT/FORELAND                 13
SALT/PASSIVE MARGIN           10
INVERSION/RIFT                 9
INVERSION/BACKARC              8
DELTA/PASSIVE MARGIN           7
SALT/INVERSION/RIFT            6
DELTA/SALT/PASSIVE MARGIN      5
SUB-SALT/FORELAND              4
THRUST/FORELAND                4
SUB-SALT/INVERSION             4
RIFT/SALT                      4
SUB-THRUST/FORELAND            4
SALT/RIFT                      3
WRENCH                         3
RIFT/INVERSION                 3
BACKARC                        3
SUB-SALT/RIFT                  3
THRUST/SUB-THRUST/FORELAND     2
INVERSION/FORELAND             2
WRENCH/FORELAND                2
SUB-THRUST                     2
FORELAND/SALT                  2
FORELAND/THRUST                2
WRENCH/DELTA                   2
WRENCH/INVERSION/BACKARC

In [229]:
train['Reservoir period'].value_counts()

Reservoir period
CRETACEOUS                78
JURASSIC                  41
NEOGENE                   39
PALEOGENE                 37
CARBONIFEROUS             29
PERMIAN                   28
DEVONIAN                  15
TRIASSIC                   8
CRETACEOUS-PALEOGENE       8
CARBONIFEROUS-PERMIAN      4
PROTEROZOIC                4
ORDOVICIAN                 3
PALEOGENE-NEOGENE          3
JURASSIC-CRETACEOUS        2
TRIASSIC-JURASSIC          2
CAMBRIAN-ORDOVICIAN        2
DEVONIAN-PERMIAN           1
CAMBRIAN                   1
DEVONIAN-CARBONIFEROUS     1
SILURIAN                   1
MESOZOIC                   1
PALEOZOIC                  1
Name: count, dtype: int64